In [0]:
dados_bronze = spark.sql("""
    SELECT 
        trans.*,
        iden.* EXCEPT (TransactionID, ingestion_timestamp, dataset_type)
    FROM fraud_detection_dev.bronze.train_transaction_raw trans
    LEFT JOIN fraud_detection_dev.bronze.train_identity_raw iden
        ON trans.TransactionID = iden.TransactionID
""")

In [0]:
from pyspark.sql import functions as F

total_rows = dados_bronze.count()
schema = dados_bronze.schema

stats_list = [
    {
        'coluna': field.name,
        'tipo': str(field.dataType).replace('Type()', '').replace('Type', ''),
        'nulos': dados_bronze.filter(F.col(field.name).isNull()).count(),
        'taxa_nulos_%': round((dados_bronze.filter(F.col(field.name).isNull()).count() / total_rows) * 100, 2),
        'valores_unicos': dados_bronze.select(field.name).distinct().count(),
        'cardinalidade_%': round((dados_bronze.select(field.name).distinct().count() / total_rows) * 100, 2)
    }
    for field in schema.fields
]

df_stats = spark.createDataFrame(stats_list)
df_stats_sorted = df_stats.orderBy(F.desc('taxa_nulos_%'))

df_stats_sorted.limit(30).display()

In [0]:
(
    df_stats_sorted
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_detection_dev.bronze.data_quality_stats")
)

In [0]:
cols_to_drop = [
    row["coluna"]
    for row in df_stats.filter(F.col("taxa_nulos_%") > 70).select("coluna").collect()
]

dados_bronze_clean = dados_bronze.drop(*cols_to_drop)

In [0]:
from pyspark.sql import functions as F

dados_bronze_clean = dados_bronze_clean.withColumnRenamed("isFraud", "target")

if "id_34" in dados_bronze_clean.columns:
    extracted = F.regexp_extract(F.col("id_34"), r":(\d+)", 1)
    dados_bronze_clean = dados_bronze_clean.withColumn(
        "id_34",
        F.when(extracted != "", extracted.cast("float")).otherwise(None)
    )

In [0]:
(
    dados_bronze_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fraud_detection_dev.silver.fraud_data_clean")
)

spark.sql("ANALYZE TABLE fraud_detection_dev.silver.fraud_data_clean COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("OPTIMIZE fraud_detection_dev.silver.fraud_data_clean ZORDER BY (TransactionID)")